# YOLOv8 Training for Shoplifting Detection
## Multi-Model Training & Ensemble Approach

This notebook trains multiple YOLOv8 models on the merged shoplifting dataset and implements ensemble techniques for maximum accuracy.

**Models to train:**
- YOLOv8n (Nano) - Fastest, baseline
- YOLOv8s (Small) - Good balance
- YOLOv8m (Medium) - Higher accuracy
- Ensemble of all models for best results

**Features:**
1. Hyperparameter tuning
2. Training with augmentation
3. Model comparison
4. Ensemble predictions
5. Performance metrics
6. Result visualization

## 1. Setup & Installation

In [ ]:
# Install required packages
!pip install ultralytics -q
!pip install opencv-python>=4.12.0 -q
!pip install pandas matplotlib seaborn pillow -q

print("✓ All packages installed successfully!")

In [ ]:
import os
import yaml
import json
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
from ultralytics import YOLO
import torch

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

print("\n✓ Libraries imported successfully!")

## 2. Configuration & Paths

In [ ]:
# Paths
BASE_DIR = Path(r'c:\Users\NIlEUN\Downloads\data_mix')
DATA_DIR = BASE_DIR / 'merged_shoplifting_dataset'
DATA_YAML = DATA_DIR / 'data.yaml'
RESULTS_DIR = BASE_DIR / 'training_results'
RESULTS_DIR.mkdir(exist_ok=True)

# Verify dataset exists
assert DATA_YAML.exists(), f"Dataset not found at {DATA_YAML}. Please run data preprocessing first!"

print(f"Dataset location: {DATA_DIR}")
print(f"Results will be saved to: {RESULTS_DIR}")
print(f"\n✓ Paths configured!")

In [ ]:
# Training configuration
TRAINING_CONFIG = {
    'epochs': 100,
    'imgsz': 640,
    'batch': 16,  # Adjust based on your GPU memory
    'patience': 20,  # Early stopping patience
    'device': 0 if torch.cuda.is_available() else 'cpu',
    'workers': 8,
    'optimizer': 'AdamW',
    'lr0': 0.001,  # Initial learning rate
    'lrf': 0.01,   # Final learning rate factor
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3,
    'warmup_momentum': 0.8,
    'warmup_bias_lr': 0.1,
    'cos_lr': True,  # Cosine learning rate scheduler
    'close_mosaic': 10,  # Disable mosaic augmentation in last N epochs
}

# Augmentation configuration (already set in data.yaml, but can override)
AUGMENTATION_CONFIG = {
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'degrees': 10.0,
    'translate': 0.1,
    'scale': 0.5,
    'shear': 2.0,
    'perspective': 0.0001,
    'flipud': 0.0,
    'fliplr': 0.5,
    'mosaic': 1.0,
    'mixup': 0.1,
    'copy_paste': 0.0,
}

print("Training Configuration:")
for key, value in TRAINING_CONFIG.items():
    print(f"  {key}: {value}")

print("\n✓ Configuration loaded!")

## 3. Load and Verify Dataset

In [ ]:
# Load dataset configuration
with open(DATA_YAML, 'r') as f:
    dataset_config = yaml.safe_load(f)

print("Dataset Configuration:")
print(f"  Classes: {dataset_config['nc']}")
print(f"  Names: {dataset_config['names']}")
print(f"  Train images: {dataset_config['splits']['train']}")
print(f"  Validation images: {dataset_config['splits']['valid']}")
print(f"  Test images: {dataset_config['splits']['test']}")

# Calculate class distribution
class_dist = dataset_config['class_distribution']
print(f"\nClass Distribution:")
for class_name, count in class_dist.items():
    print(f"  {class_name}: {count}")

print("\n✓ Dataset verified!")

## 4. Model Training - YOLOv8n (Nano)

In [ ]:
# Train YOLOv8n (Nano) - Fastest model
print("=" * 60)
print("Training YOLOv8n (Nano) - Fastest, Baseline Model")
print("=" * 60)

model_n = YOLO('yolov8n.pt')

results_n = model_n.train(
    data=str(DATA_YAML),
    epochs=TRAINING_CONFIG['epochs'],
    imgsz=TRAINING_CONFIG['imgsz'],
    batch=TRAINING_CONFIG['batch'],
    patience=TRAINING_CONFIG['patience'],
    device=TRAINING_CONFIG['device'],
    workers=TRAINING_CONFIG['workers'],
    optimizer=TRAINING_CONFIG['optimizer'],
    lr0=TRAINING_CONFIG['lr0'],
    lrf=TRAINING_CONFIG['lrf'],
    momentum=TRAINING_CONFIG['momentum'],
    weight_decay=TRAINING_CONFIG['weight_decay'],
    warmup_epochs=TRAINING_CONFIG['warmup_epochs'],
    warmup_momentum=TRAINING_CONFIG['warmup_momentum'],
    warmup_bias_lr=TRAINING_CONFIG['warmup_bias_lr'],
    cos_lr=TRAINING_CONFIG['cos_lr'],
    close_mosaic=TRAINING_CONFIG['close_mosaic'],
    name='yolov8n_shoplifting',
    project=str(RESULTS_DIR),
    exist_ok=True,
    pretrained=True,
    verbose=True,
    save=True,
    save_period=10,  # Save checkpoint every 10 epochs
    plots=True,
    **AUGMENTATION_CONFIG
)

print("\n✓ YOLOv8n training complete!")

## 5. Model Training - YOLOv8s (Small)

In [ ]:
# Train YOLOv8s (Small) - Good balance between speed and accuracy
print("=" * 60)
print("Training YOLOv8s (Small) - Balanced Speed & Accuracy")
print("=" * 60)

model_s = YOLO('yolov8s.pt')

results_s = model_s.train(
    data=str(DATA_YAML),
    epochs=TRAINING_CONFIG['epochs'],
    imgsz=TRAINING_CONFIG['imgsz'],
    batch=TRAINING_CONFIG['batch'],
    patience=TRAINING_CONFIG['patience'],
    device=TRAINING_CONFIG['device'],
    workers=TRAINING_CONFIG['workers'],
    optimizer=TRAINING_CONFIG['optimizer'],
    lr0=TRAINING_CONFIG['lr0'],
    lrf=TRAINING_CONFIG['lrf'],
    momentum=TRAINING_CONFIG['momentum'],
    weight_decay=TRAINING_CONFIG['weight_decay'],
    warmup_epochs=TRAINING_CONFIG['warmup_epochs'],
    warmup_momentum=TRAINING_CONFIG['warmup_momentum'],
    warmup_bias_lr=TRAINING_CONFIG['warmup_bias_lr'],
    cos_lr=TRAINING_CONFIG['cos_lr'],
    close_mosaic=TRAINING_CONFIG['close_mosaic'],
    name='yolov8s_shoplifting',
    project=str(RESULTS_DIR),
    exist_ok=True,
    pretrained=True,
    verbose=True,
    save=True,
    save_period=10,
    plots=True,
    **AUGMENTATION_CONFIG
)

print("\n✓ YOLOv8s training complete!")

## 6. Model Training - YOLOv8m (Medium)

In [ ]:
# Train YOLOv8m (Medium) - Higher accuracy, slower
print("=" * 60)
print("Training YOLOv8m (Medium) - Higher Accuracy Model")
print("=" * 60)

model_m = YOLO('yolov8m.pt')

results_m = model_m.train(
    data=str(DATA_YAML),
    epochs=TRAINING_CONFIG['epochs'],
    imgsz=TRAINING_CONFIG['imgsz'],
    batch=max(8, TRAINING_CONFIG['batch'] // 2),  # Reduce batch size for larger model
    patience=TRAINING_CONFIG['patience'],
    device=TRAINING_CONFIG['device'],
    workers=TRAINING_CONFIG['workers'],
    optimizer=TRAINING_CONFIG['optimizer'],
    lr0=TRAINING_CONFIG['lr0'],
    lrf=TRAINING_CONFIG['lrf'],
    momentum=TRAINING_CONFIG['momentum'],
    weight_decay=TRAINING_CONFIG['weight_decay'],
    warmup_epochs=TRAINING_CONFIG['warmup_epochs'],
    warmup_momentum=TRAINING_CONFIG['warmup_momentum'],
    warmup_bias_lr=TRAINING_CONFIG['warmup_bias_lr'],
    cos_lr=TRAINING_CONFIG['cos_lr'],
    close_mosaic=TRAINING_CONFIG['close_mosaic'],
    name='yolov8m_shoplifting',
    project=str(RESULTS_DIR),
    exist_ok=True,
    pretrained=True,
    verbose=True,
    save=True,
    save_period=10,
    plots=True,
    **AUGMENTATION_CONFIG
)

print("\n✓ YOLOv8m training complete!")

## 7. Model Evaluation & Comparison

In [ ]:
# Load trained models
model_n_trained = YOLO(RESULTS_DIR / 'yolov8n_shoplifting' / 'weights' / 'best.pt')
model_s_trained = YOLO(RESULTS_DIR / 'yolov8s_shoplifting' / 'weights' / 'best.pt')
model_m_trained = YOLO(RESULTS_DIR / 'yolov8m_shoplifting' / 'weights' / 'best.pt')

print("✓ Trained models loaded!")

In [ ]:
# Validate all models on test set
print("=" * 60)
print("Evaluating Models on Test Set")
print("=" * 60)

# YOLOv8n evaluation
print("\nEvaluating YOLOv8n...")
metrics_n = model_n_trained.val(
    data=str(DATA_YAML),
    split='test',
    batch=TRAINING_CONFIG['batch'],
    imgsz=TRAINING_CONFIG['imgsz'],
    device=TRAINING_CONFIG['device']
)

# YOLOv8s evaluation
print("\nEvaluating YOLOv8s...")
metrics_s = model_s_trained.val(
    data=str(DATA_YAML),
    split='test',
    batch=TRAINING_CONFIG['batch'],
    imgsz=TRAINING_CONFIG['imgsz'],
    device=TRAINING_CONFIG['device']
)

# YOLOv8m evaluation
print("\nEvaluating YOLOv8m...")
metrics_m = model_m_trained.val(
    data=str(DATA_YAML),
    split='test',
    batch=max(8, TRAINING_CONFIG['batch'] // 2),
    imgsz=TRAINING_CONFIG['imgsz'],
    device=TRAINING_CONFIG['device']
)

print("\n✓ Model evaluation complete!")

In [ ]:
# Create comparison table
comparison_data = {
    'Model': ['YOLOv8n', 'YOLOv8s', 'YOLOv8m'],
    'mAP50': [
        metrics_n.box.map50,
        metrics_s.box.map50,
        metrics_m.box.map50
    ],
    'mAP50-95': [
        metrics_n.box.map,
        metrics_s.box.map,
        metrics_m.box.map
    ],
    'Precision': [
        metrics_n.box.mp,
        metrics_s.box.mp,
        metrics_m.box.mp
    ],
    'Recall': [
        metrics_n.box.mr,
        metrics_s.box.mr,
        metrics_m.box.mr
    ],
    'F1-Score': [
        2 * (metrics_n.box.mp * metrics_n.box.mr) / (metrics_n.box.mp + metrics_n.box.mr + 1e-10),
        2 * (metrics_s.box.mp * metrics_s.box.mr) / (metrics_s.box.mp + metrics_s.box.mr + 1e-10),
        2 * (metrics_m.box.mp * metrics_m.box.mr) / (metrics_m.box.mp + metrics_m.box.mr + 1e-10)
    ]
}

df_comparison = pd.DataFrame(comparison_data)
print("\n" + "=" * 60)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 60)
print(df_comparison.to_string(index=False))

# Save comparison
df_comparison.to_csv(RESULTS_DIR / 'model_comparison.csv', index=False)
print(f"\n✓ Comparison saved to {RESULTS_DIR / 'model_comparison.csv'}")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
metrics_to_plot = ['mAP50', 'mAP50-95', 'Precision', 'Recall', 'F1-Score']
colors = ['#3498db', '#2ecc71', '#e74c3c']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx // 3, idx % 3]
    bars = ax.bar(df_comparison['Model'], df_comparison[metric], color=colors, alpha=0.7)
    ax.set_title(f'{metric}', fontsize=14, fontweight='bold')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.0)
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for i, bar in enumerate(bars):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

# Hide the last subplot
axes[1, 2].axis('off')

plt.suptitle('YOLOv8 Models Performance Comparison', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Comparison visualization saved!")

## 8. Ensemble Predictions (Weighted Voting)

In [ ]:
def ensemble_predict(image_path, models, weights=None, conf_threshold=0.25, iou_threshold=0.45):
    """
    Ensemble prediction using weighted voting
    
    Args:
        image_path: Path to input image
        models: List of YOLO models
        weights: List of weights for each model (based on mAP50)
        conf_threshold: Confidence threshold
        iou_threshold: IoU threshold for NMS
    """
    if weights is None:
        # Use mAP50 as weights
        weights = [
            metrics_n.box.map50,
            metrics_s.box.map50,
            metrics_m.box.map50
        ]
    
    # Normalize weights
    weights = np.array(weights)
    weights = weights / weights.sum()
    
    # Get predictions from all models
    all_boxes = []
    all_scores = []
    all_classes = []
    
    for model, weight in zip(models, weights):
        results = model.predict(
            image_path,
            conf=conf_threshold,
            iou=iou_threshold,
            verbose=False
        )[0]
        
        if len(results.boxes) > 0:
            boxes = results.boxes.xyxy.cpu().numpy()
            scores = results.boxes.conf.cpu().numpy() * weight  # Weight the scores
            classes = results.boxes.cls.cpu().numpy()
            
            all_boxes.append(boxes)
            all_scores.append(scores)
            all_classes.append(classes)
    
    if not all_boxes:
        return None, None, None
    
    # Concatenate all predictions
    all_boxes = np.vstack(all_boxes)
    all_scores = np.concatenate(all_scores)
    all_classes = np.concatenate(all_classes)
    
    # Apply NMS to remove duplicates
    from ultralytics.utils.ops import non_max_suppression
    
    # Convert to torch tensors for NMS
    boxes_tensor = torch.from_numpy(all_boxes).float()
    scores_tensor = torch.from_numpy(all_scores).float()
    classes_tensor = torch.from_numpy(all_classes).float()
    
    # Combine boxes, scores, and classes
    predictions = torch.cat([
        boxes_tensor,
        scores_tensor.unsqueeze(1),
        classes_tensor.unsqueeze(1)
    ], dim=1).unsqueeze(0)
    
    # Apply NMS
    nms_predictions = non_max_suppression(
        predictions,
        conf_thres=conf_threshold,
        iou_thres=iou_threshold
    )[0]
    
    if nms_predictions is None or len(nms_predictions) == 0:
        return None, None, None
    
    final_boxes = nms_predictions[:, :4].cpu().numpy()
    final_scores = nms_predictions[:, 4].cpu().numpy()
    final_classes = nms_predictions[:, 5].cpu().numpy()
    
    return final_boxes, final_scores, final_classes

print("✓ Ensemble prediction function defined!")

## 9. Test Ensemble on Sample Images

In [ ]:
# Get sample test images
test_images_dir = DATA_DIR / 'test' / 'images'
sample_images = list(test_images_dir.glob('*'))[:6]

models_list = [model_n_trained, model_s_trained, model_m_trained]
class_names = dataset_config['names']
colors = [(0, 255, 0), (255, 0, 0)]  # Green for normal, Red for theft

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, img_path in enumerate(sample_images):
    if idx >= len(axes):
        break
    
    # Get ensemble predictions
    boxes, scores, classes = ensemble_predict(str(img_path), models_list)
    
    # Load and display image
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    if boxes is not None:
        for box, score, cls in zip(boxes, scores, classes):
            x1, y1, x2, y2 = map(int, box)
            class_id = int(cls)
            color = colors[class_id] if class_id < len(colors) else (255, 255, 0)
            
            # Draw box
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            
            # Draw label
            label = f"{class_names[class_id]}: {score:.2f}"
            cv2.putText(img, label, (x1, y1 - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    
    axes[idx].imshow(img)
    axes[idx].set_title(f'Ensemble: {img_path.name[:30]}...', fontsize=10)
    axes[idx].axis('off')

plt.suptitle('Ensemble Model Predictions on Test Set', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'ensemble_predictions.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Ensemble predictions visualized!")

## 10. Generate Final Report

In [ ]:
# Generate comprehensive training report
report = f"""# YOLOV8 TRAINING REPORT - SHOPLIFTING DETECTION
Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

## TRAINING SUMMARY

Successfully trained 3 YOLOv8 models on the merged shoplifting detection dataset.

### Models Trained:
1. **YOLOv8n (Nano)** - Fastest, baseline model
2. **YOLOv8s (Small)** - Balanced speed and accuracy
3. **YOLOv8m (Medium)** - Highest accuracy

### Training Configuration:
- Epochs: {TRAINING_CONFIG['epochs']}
- Image Size: {TRAINING_CONFIG['imgsz']}x{TRAINING_CONFIG['imgsz']}
- Batch Size: {TRAINING_CONFIG['batch']}
- Optimizer: {TRAINING_CONFIG['optimizer']}
- Device: {'GPU' if torch.cuda.is_available() else 'CPU'}
- Early Stopping Patience: {TRAINING_CONFIG['patience']} epochs

### Dataset Statistics:
- Training Images: {dataset_config['splits']['train']}
- Validation Images: {dataset_config['splits']['valid']}
- Test Images: {dataset_config['splits']['test']}
- Classes: {dataset_config['nc']} ({', '.join(dataset_config['names'])})

## PERFORMANCE METRICS (Test Set)

### YOLOv8n (Nano):
- mAP@50: {metrics_n.box.map50:.4f}
- mAP@50-95: {metrics_n.box.map:.4f}
- Precision: {metrics_n.box.mp:.4f}
- Recall: {metrics_n.box.mr:.4f}
- F1-Score: {2 * (metrics_n.box.mp * metrics_n.box.mr) / (metrics_n.box.mp + metrics_n.box.mr + 1e-10):.4f}

### YOLOv8s (Small):
- mAP@50: {metrics_s.box.map50:.4f}
- mAP@50-95: {metrics_s.box.map:.4f}
- Precision: {metrics_s.box.mp:.4f}
- Recall: {metrics_s.box.mr:.4f}
- F1-Score: {2 * (metrics_s.box.mp * metrics_s.box.mr) / (metrics_s.box.mp + metrics_s.box.mr + 1e-10):.4f}

### YOLOv8m (Medium):
- mAP@50: {metrics_m.box.map50:.4f}
- mAP@50-95: {metrics_m.box.map:.4f}
- Precision: {metrics_m.box.mp:.4f}
- Recall: {metrics_m.box.mr:.4f}
- F1-Score: {2 * (metrics_m.box.mp * metrics_m.box.mr) / (metrics_m.box.mp + metrics_m.box.mr + 1e-10):.4f}

## BEST MODEL

Based on mAP@50, the best performing model is: **{df_comparison.loc[df_comparison['mAP50'].idxmax(), 'Model']}**

## ENSEMBLE APPROACH

Implemented weighted ensemble using all three models with weights based on their mAP@50 scores.
This approach combines predictions from multiple models for improved robustness and accuracy.

### Ensemble Weights:
- YOLOv8n: {metrics_n.box.map50 / (metrics_n.box.map50 + metrics_s.box.map50 + metrics_m.box.map50):.4f}
- YOLOv8s: {metrics_s.box.map50 / (metrics_n.box.map50 + metrics_s.box.map50 + metrics_m.box.map50):.4f}
- YOLOv8m: {metrics_m.box.map50 / (metrics_n.box.map50 + metrics_s.box.map50 + metrics_m.box.map50):.4f}

## MODEL LOCATIONS

Trained models saved in:
```
{RESULTS_DIR}
+-- yolov8n_shoplifting/
|   +-- weights/
|       +-- best.pt
|       +-- last.pt
+-- yolov8s_shoplifting/
|   +-- weights/
|       +-- best.pt
|       +-- last.pt
+-- yolov8m_shoplifting/
    +-- weights/
        +-- best.pt
        +-- last.pt
```

## USAGE EXAMPLES

### Single Model Inference:
```python
from ultralytics import YOLO

# Load best model
model = YOLO('{RESULTS_DIR}/yolov8s_shoplifting/weights/best.pt')

# Predict on image
results = model.predict('path/to/image.jpg', conf=0.25)

# Display results
results[0].show()
```

### Ensemble Inference:
```python
# Use the ensemble_predict function from this notebook
boxes, scores, classes = ensemble_predict(
    'path/to/image.jpg',
    [model_n, model_s, model_m]
)
```

## NEXT STEPS

1. **Deploy best model** for real-time inference
2. **Fine-tune hyperparameters** if needed
3. **Collect more data** for classes with lower performance
4. **Implement model optimization** (TensorRT, ONNX) for faster inference
5. **Set up monitoring** for production deployment

---

**Training completed successfully!**
"""

# Save report
with open(RESULTS_DIR / 'TRAINING_REPORT.md', 'w', encoding='utf-8') as f:
    f.write(report)

print(report)
print(f"\n✓ Report saved to: {RESULTS_DIR / 'TRAINING_REPORT.md'}")

## 11. Export Models for Deployment

In [ ]:
# Export best model to different formats for deployment
best_model_name = df_comparison.loc[df_comparison['mAP50'].idxmax(), 'Model'].lower()
best_model_path = RESULTS_DIR / f'{best_model_name}_shoplifting' / 'weights' / 'best.pt'
best_model = YOLO(best_model_path)

export_dir = RESULTS_DIR / 'exported_models'
export_dir.mkdir(exist_ok=True)

print(f"Exporting best model ({best_model_name.upper()}) to multiple formats...\n")

# Export to ONNX (cross-platform)
try:
    print("Exporting to ONNX...")
    best_model.export(format='onnx', simplify=True)
    print("✓ ONNX export complete")
except Exception as e:
    print(f"✗ ONNX export failed: {e}")

# Export to TorchScript (PyTorch deployment)
try:
    print("\nExporting to TorchScript...")
    best_model.export(format='torchscript')
    print("✓ TorchScript export complete")
except Exception as e:
    print(f"✗ TorchScript export failed: {e}")

# Export to TFLite (mobile/edge devices)
try:
    print("\nExporting to TFLite...")
    best_model.export(format='tflite')
    print("✓ TFLite export complete")
except Exception as e:
    print(f"✗ TFLite export failed: {e}")

print("\n✓ Model export process complete!")
print(f"Exported models location: Check the model directory")

## Summary

This notebook has:
1. ✅ Trained 3 YOLOv8 models (Nano, Small, Medium)
2. ✅ Evaluated all models on test set
3. ✅ Compared model performances
4. ✅ Implemented ensemble predictions
5. ✅ Generated comprehensive training report
6. ✅ Exported best model for deployment

**All models and results are saved in the `training_results` directory!**